# CHƯƠNG 3 – MỤC 3.1: PHÂN TÍCH MÔ TẢ VÀ KHÁM PHÁ DỮ LIỆU (EDA)

**Notebook này chạy tiếp sau `Feature_engirneering.ipynb`.**  
Đọc thẳng file `olist_final_dataset_labeled.csv` đã xuất ra ở bước feature engineering,  
không tính lại biến từ đầu.

### Biến đầu vào (`df_final_labeled`):
| Tên biến raw | Tên biến norm | Ý nghĩa |
|---|---|---|
| `raw_delivery_delay` | `norm_delivery_delay` | Độ lệch giao hàng so với ước tính (âm = trễ, dương = sớm) |
| `raw_lead_time` | `norm_lead_time` | Tổng thời gian đặt → nhận hàng (ngày) |
| `raw_seller_speed` | `norm_seller_speed` | Thời gian người bán chuẩn bị hàng (ngày) |
| `raw_freight_ratio` | `norm_freight_ratio` | Tỷ lệ phí vận chuyển / giá trị hàng |
| `raw_installments` | `norm_installments` | Số kỳ trả góp lớn nhất |
| `raw_price` | `norm_price` | Tổng giá trị hàng hóa (BRL, clip 0–2000) |
| `raw_desc_length` | `norm_desc_length` | Độ dài mô tả sản phẩm trung bình (ký tự) |
| `raw_photos_qty` | `norm_photos_qty` | Số ảnh sản phẩm trung bình |
| `raw_seller_rep` | `norm_seller_rep` | Danh tiếng người bán (điểm TB tích lũy) |
| `raw_customer_order_count` | `norm_customer_order_count` | Số lần mua tích lũy (cumcount, 0-indexed) |
| – | `review_score` | Điểm đánh giá gốc (1–5) |
| – | `review_binary` | Nhãn phân loại: 0 = không hài lòng (≤3), 1 = hài lòng (>3) |

---

## 0. Import & Đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kruskal
import warnings
warnings.filterwarnings('ignore')

# ── Cấu hình đồ thị ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
})
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Màu sắc nhất quán ────────────────────────────────────────────────────────
SCORE_COLORS = {1:'#d62728', 2:'#ff7f0e', 3:'#bcbd22', 4:'#2ca02c', 5:'#1f77b4'}
GROUP_COLORS = {
    'Giao hàng': '#1f77b4',
    'Giao dịch': '#ff7f0e',
    'Sản phẩm':  '#2ca02c',
    'Lịch sử':   '#9467bd'
}
GROUP_MAP = {
    'raw_delivery_delay':       'Giao hàng',
    'raw_lead_time':            'Giao hàng',
    'raw_seller_speed':         'Giao hàng',
    'raw_freight_ratio':        'Giao dịch',
    'raw_installments':         'Giao dịch',
    'raw_price':                'Giao dịch',
    'raw_desc_length':          'Sản phẩm',
    'raw_photos_qty':           'Sản phẩm',
    'raw_seller_rep':           'Lịch sử',
    'raw_customer_order_count': 'Lịch sử',
}
LABEL = {
    'raw_delivery_delay':       'delivery_delay (ngày)',
    'raw_lead_time':            'lead_time (ngày)',
    'raw_seller_speed':         'seller_speed (ngày)',
    'raw_freight_ratio':        'freight_ratio',
    'raw_installments':         'installments (kỳ)',
    'raw_price':                'price (BRL)',
    'raw_desc_length':          'desc_length (ký tự)',
    'raw_photos_qty':           'photos_qty (ảnh)',
    'raw_seller_rep':           'seller_rep (điểm TB)',
    'raw_customer_order_count': 'customer_order_count',
}

RAW_COLS  = list(LABEL.keys())
NORM_COLS = [c.replace('raw_', 'norm_') for c in RAW_COLS]

print('✓ Thư viện đã sẵn sàng')

In [ ]:
# ── Đọc file đã xuất từ Feature_engirneering.ipynb ───────────────────────────
df = pd.read_csv('olist_final_dataset_labeled.csv')

print(f'Shape: {df.shape}')
print(f'Cột: {list(df.columns)}')
df.head(3)

In [ ]:
# ── Kiểm tra nhanh ───────────────────────────────────────────────────────────
print('=== Missing values ===')
print(df[RAW_COLS + ['review_score','review_binary']].isna().sum())
print(f'\nPhân phối review_binary:')
print(df['review_binary'].value_counts())
print(f'\nPhân phối review_score:')
print(df['review_score'].value_counts().sort_index())

---
## 3.1.1. Phân phối mức độ hài lòng

In [ ]:
# ── Thống kê mô tả review_score ──────────────────────────────────────────────
sc = df['review_score']
freq = sc.value_counts().sort_index()
pct  = sc.value_counts(normalize=True).sort_index() * 100
cum  = pct.cumsum()

print('=' * 55)
print('THỐNG KÊ MÔ TẢ – review_score')
print('=' * 55)
print(sc.describe().round(4).to_string())
print(f'\nYếu vị (Mode):  {int(sc.mode()[0])}')
print(f'Skewness:       {sc.skew():.4f}')
print(f'Kurtosis:       {sc.kurtosis():.4f}')
print()

tbl = pd.DataFrame({
    'Điểm': freq.index.astype(int),
    'Tần số': freq.values,
    'Tỷ lệ (%)': pct.round(2).values,
    'Tích lũy (%)': cum.round(2).values
})
print('Bảng tần suất:')
print(tbl.to_string(index=False))
print()
sat_pct = pct[pct.index >= 4].sum()
print(f'→ Tỷ lệ HÀI LÒNG (4-5 sao): {sat_pct:.2f}%')
print(f'→ Tỷ lệ KHÔNG HÀI LÒNG (1-3 sao): {100-sat_pct:.2f}%')

In [ ]:
# ── Hình 3.1: Phân phối review_score ─────────────────────────────────────────
freq = df['review_score'].value_counts().sort_index()
pct  = df['review_score'].value_counts(normalize=True).sort_index() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── (a) Biểu đồ cột
ax = axes[0]
bars = ax.bar(
    freq.index, freq.values,
    color=[SCORE_COLORS[int(i)] for i in freq.index],
    edgecolor='white', linewidth=0.9, width=0.6
)
for bar, p in zip(bars, pct.values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 300,
        f'{p:.1f}%', ha='center', va='bottom',
        fontsize=11, fontweight='bold'
    )
ax.set_xlabel('Điểm đánh giá (review_score)', fontsize=11)
ax.set_ylabel('Số đơn hàng', fontsize=11)
ax.set_title('(a) Tần suất theo điểm đánh giá', fontsize=11, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xticks([1, 2, 3, 4, 5])

# ── (b) Donut chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    freq.values,
    labels=[f'Điểm {int(i)}' for i in freq.index],
    autopct='%1.1f%%',
    colors=[SCORE_COLORS[int(i)] for i in freq.index],
    startangle=140, pctdistance=0.80,
    wedgeprops=dict(edgecolor='white', linewidth=1.5, width=0.58)
)
for at in autotexts:
    at.set_fontsize(9)
ax2.set_title('(b) Tỷ lệ phần trăm từng mức điểm', fontsize=11, fontweight='bold')

plt.suptitle(
    'Hình 3.1 – Phân phối điểm đánh giá (review_score)\n'
    f'N = {len(df):,} đơn hàng đã giao thành công | Mean = {df["review_score"].mean():.2f} | Skew = {df["review_score"].skew():.2f}',
    fontsize=11, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('hinh3_1_phan_phoi_review_score.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_1_phan_phoi_review_score.png')

In [ ]:
# ── Hình 3.2: Phân loại nhị phân + xu hướng theo tháng ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── (a) Mất cân bằng lớp
binary_cnt = df['review_binary'].value_counts().sort_index()
binary_pct = binary_cnt / binary_cnt.sum() * 100
labels_b   = ['Không hài lòng\n(score ≤ 3)', 'Hài lòng\n(score > 3)']
colors_b   = ['#d62728', '#2ca02c']

bars_b = axes[0].bar(
    labels_b, binary_cnt.values,
    color=colors_b, edgecolor='white', width=0.45
)
for bar, cnt, p in zip(bars_b, binary_cnt.values, binary_pct.values):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 400,
        f'{cnt:,}\n({p:.1f}%)',
        ha='center', fontsize=11, fontweight='bold'
    )
axes[0].set_ylabel('Số đơn hàng', fontsize=11)
axes[0].set_title('(a) Phân loại nhị phân (review_binary)', fontsize=11, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ratio = binary_cnt[1] / binary_cnt[0]
axes[0].text(0.5, 0.95, f'Tỷ lệ mất cân bằng: {ratio:.1f}:1',
             transform=axes[0].transAxes, ha='center', va='top',
             fontsize=10, color='#555555',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffde7', edgecolor='#f0c040'))

# ── (b) Điểm TB theo tháng (cần order_purchase_timestamp)
# Vì file labeled không có cột timestamp → dùng index làm proxy hoặc bỏ qua
# Nếu bạn có file orders, merge vào:
try:
    orders_ts = pd.read_csv('olist_orders_dataset_clean.csv',
                            usecols=['order_id','order_purchase_timestamp'])
    orders_ts['order_purchase_timestamp'] = pd.to_datetime(
        orders_ts['order_purchase_timestamp'], errors='coerce')
    df_t = df.merge(orders_ts, on='order_id', how='left')
    df_t['ym'] = df_t['order_purchase_timestamp'].dt.to_period('M')
    monthly = df_t.groupby('ym').agg(
        avg=('review_score','mean'), n=('review_score','count')
    ).reset_index()
    monthly = monthly[monthly['n'] >= 100]
    monthly['ym_str'] = monthly['ym'].astype(str)
    x = range(len(monthly))
    axes[1].plot(x, monthly['avg'], color='#1f77b4', lw=2.2, marker='o', ms=4)
    axes[1].fill_between(x, monthly['avg'], alpha=0.12, color='#1f77b4')
    axes[1].set_ylim(3.5, 5.0)
    ax2r = axes[1].twinx()
    ax2r.bar(x, monthly['n'], color='#aec7e8', alpha=0.35)
    ax2r.set_ylabel('Số đơn hàng', fontsize=10, color='#5b8db8')
    ax2r.tick_params(axis='y', labelcolor='#5b8db8')
    step = max(1, len(monthly)//10)
    axes[1].set_xticks(list(x)[::step])
    axes[1].set_xticklabels(monthly['ym_str'].tolist()[::step], rotation=40, ha='right', fontsize=9)
    axes[1].set_ylabel('Điểm TB', fontsize=11)
    axes[1].set_title('(b) Xu hướng điểm đánh giá TB theo tháng', fontsize=11, fontweight='bold')
except Exception as e:
    axes[1].text(0.5, 0.5, f'Cần file orders để vẽ xu hướng\n({e})',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=10)
    axes[1].set_title('(b) Xu hướng theo tháng', fontsize=11)

plt.suptitle('Hình 3.2 – Phân loại nhị phân và xu hướng điểm hài lòng',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('hinh3_2_binary_va_xuhuong.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_2_binary_va_xuhuong.png')

---
## 3.1.2. Tương quan giữa các biến chuẩn hóa với điểm số hài lòng

In [ ]:
# ── Bảng 3.1: Thống kê mô tả 10 biến raw ─────────────────────────────────────
desc = df[RAW_COLS + ['review_score']].describe().T
desc.index = [LABEL.get(i, i) for i in desc.index[:-1]] + ['review_score (1–5)']
desc = desc[['count','mean','std','min','50%','max']]
desc.columns = ['N', 'Trung bình', 'Std', 'Min', 'Median', 'Max']

print('BẢNG 3.1 – Thống kê mô tả 10 biến đặc trưng và biến mục tiêu')
print('=' * 85)
print(desc.round(3).to_string())

In [ ]:
# ── Bảng 3.2: Tương quan Pearson & Spearman + Kruskal-Wallis ─────────────────
rows = []
for raw in RAW_COLS:
    norm = raw.replace('raw_', 'norm_')
    x = df[raw].dropna()
    y = df.loc[x.index, 'review_score']

    r_p, pv_p = pearsonr(x, y)
    r_s, pv_s = spearmanr(x, y)

    # Kruskal-Wallis
    groups = [df[df['review_score'] == s][raw].dropna() for s in [1, 2, 3, 4, 5]]
    groups = [g for g in groups if len(g) > 0]
    H, pv_kw = kruskal(*groups)

    sig = '***' if pv_p < 0.001 else ('**' if pv_p < 0.01 else ('*' if pv_p < 0.05 else 'ns'))
    rows.append({
        'Biến': raw.replace('raw_',''),
        'Nhóm': GROUP_MAP[raw],
        'Pearson r': round(r_p, 4),
        'Spearman ρ': round(r_s, 4),
        'KW H-stat': round(H, 1),
        'p-value (Pearson)': f'{pv_p:.2e}',
        'Ý nghĩa': sig,
    })

corr_df = pd.DataFrame(rows).sort_values('Pearson r')
print('BẢNG 3.2 – Tương quan với review_score')
print('Ghi chú: *** p<0.001 | ** p<0.01 | * p<0.05 | ns = không ý nghĩa')
print('=' * 85)
print(corr_df.to_string(index=False))

In [ ]:
# ── Hình 3.3: Lollipop chart tương quan Pearson theo nhóm biến ───────────────
corr_sorted = corr_df.sort_values('Pearson r').copy()
corr_sorted['color'] = corr_sorted['Nhóm'].map(GROUP_COLORS)

fig, ax = plt.subplots(figsize=(11, 6))

for _, row in corr_sorted.iterrows():
    ax.plot([0, row['Pearson r']], [row['Biến'], row['Biến']],
            color=row['color'], lw=3, alpha=0.8, solid_capstyle='round')
    ax.scatter(row['Pearson r'], row['Biến'],
               color=row['color'], s=120, zorder=5, edgecolors='white', linewidth=0.8)
    xoff = 0.005 if row['Pearson r'] >= 0 else -0.005
    ha   = 'left' if row['Pearson r'] >= 0 else 'right'
    label_txt = f"{row['Pearson r']:+.3f}  {row['Ý nghĩa']}"
    ax.text(row['Pearson r'] + xoff, row['Biến'], label_txt,
            va='center', ha=ha, fontsize=10, fontweight='bold',
            color=row['color'])

ax.axvline(0, color='black', lw=1, ls='--', alpha=0.5)
ax.set_xlabel('Hệ số tương quan Pearson (r) với review_score', fontsize=11)
ax.set_title('Hình 3.3 – Tương quan Pearson của 10 biến đặc trưng với review_score',
             fontsize=11, fontweight='bold')

# Vùng nền
ax.axvspan(-0.5, 0, alpha=0.04, color='red')
ax.axvspan(0, 0.5, alpha=0.04, color='green')
ax.set_xlim(-0.45, 0.50)

# Legend nhóm
handles = [mpatches.Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=handles, loc='lower right', fontsize=9,
          title='Nhóm biến', framealpha=0.9)

plt.tight_layout()
plt.savefig('hinh3_3_lollipop_tuong_quan.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_3_lollipop_tuong_quan.png')

In [ ]:
# ── Hình 3.4: Ma trận tương quan Pearson (11 biến) ───────────────────────────
cols_for_corr = RAW_COLS + ['review_score']
corr_mat = df[cols_for_corr].corr(method='pearson')

nice = {r: LABEL[r] for r in RAW_COLS}
nice['review_score'] = '★ review_score'
corr_mat.index   = [nice.get(c, c) for c in corr_mat.index]
corr_mat.columns = [nice.get(c, c) for c in corr_mat.columns]

mask = np.triu(np.ones_like(corr_mat, dtype=bool))
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr_mat, mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.75, 'label': 'Pearson r'},
    ax=ax, annot_kws={'size': 8.5}
)
ax.set_title(
    'Hình 3.4 – Ma trận tương quan Pearson: 10 biến đặc trưng + review_score',
    fontsize=11, fontweight='bold', pad=15
)
plt.tight_layout()
plt.savefig('hinh3_4_matrix_tuong_quan.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_4_matrix_tuong_quan.png')

In [ ]:
# ── Hình 3.5: Nhóm GIAO HÀNG – boxplot theo điểm đánh giá ───────────────────
delivery_vars = ['raw_delivery_delay', 'raw_lead_time', 'raw_seller_speed']
NICE_D = {
    'raw_delivery_delay': 'delivery_delay\n(âm=trễ, dương=sớm, ngày)',
    'raw_lead_time':      'lead_time\n(đặt→nhận, ngày)',
    'raw_seller_speed':   'seller_speed\n(chuẩn bị hàng, ngày)',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, var in zip(axes, delivery_vars):
    groups = [
        df[df['review_score'] == s][var].dropna()
        for s in [1, 2, 3, 4, 5]
    ]
    bp = ax.boxplot(
        groups,
        labels=[f'⭐{i}' for i in [1, 2, 3, 4, 5]],
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2.2),
        flierprops=dict(marker='o', markersize=2, alpha=0.2),
        widths=0.55
    )
    for patch, c in zip(bp['boxes'], SCORE_COLORS.values()):
        patch.set_facecolor(c)
        patch.set_alpha(0.72)

    # Đánh dấu giá trị trung bình
    means = [g.mean() for g in groups]
    ax.plot(range(1, 6), means, 'D--', color='#333333',
            ms=7, lw=1.5, label='Mean', zorder=5)
    for i, m in enumerate(means, 1):
        ax.text(i, m, f'\n{m:.1f}', ha='center', fontsize=7.5, color='#333')

    ax.set_xlabel('Điểm đánh giá', fontsize=10)
    ax.set_ylabel(NICE_D[var], fontsize=10)
    ax.set_title(NICE_D[var], fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Hình 3.5 – Nhóm Giao hàng: Phân phối biến theo điểm đánh giá (1–5 sao)',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hinh3_5_giaohang_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_5_giaohang_boxplot.png')

In [ ]:
# ── Hình 3.6: Nhóm GIAO DỊCH – bar trung bình theo điểm ─────────────────────
trans_vars = ['raw_freight_ratio', 'raw_installments', 'raw_price']
NICE_T = {
    'raw_freight_ratio':  'freight_ratio\n(phí/giá trị hàng)',
    'raw_installments':   'installments\n(số kỳ trả góp)',
    'raw_price':          'price\n(BRL, clip 0–2000)',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, var in zip(axes, trans_vars):
    means = df.groupby('review_score')[var].mean()
    stds  = df.groupby('review_score')[var].sem()

    ax.bar(
        means.index, means.values,
        color=[SCORE_COLORS[int(i)] for i in means.index],
        edgecolor='white', width=0.55, alpha=0.85,
        yerr=stds.values, capsize=4, error_kw={'linewidth': 1.2}
    )
    for s, v in zip(means.index, means.values):
        ax.text(s, v * 1.015, f'{v:.2f}',
                ha='center', fontsize=9.5, fontweight='bold')

    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xlabel('Điểm đánh giá', fontsize=10)
    ax.set_ylabel(NICE_T[var], fontsize=10)
    ax.set_title(NICE_T[var], fontsize=10, fontweight='bold')

plt.suptitle('Hình 3.6 – Nhóm Giao dịch: Giá trị trung bình (±SE) theo điểm đánh giá',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hinh3_6_giaodich_bar.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_6_giaodich_bar.png')

In [ ]:
# ── Hình 3.7: Nhóm SẢN PHẨM & LỊCH SỬ – line plot trung bình ───────────────
other_vars = ['raw_desc_length', 'raw_photos_qty', 'raw_seller_rep', 'raw_customer_order_count']
NICE_O = {
    'raw_desc_length':          'desc_length (ký tự mô tả)',
    'raw_photos_qty':           'photos_qty (số ảnh SP)',
    'raw_seller_rep':           'seller_rep (điểm TB tích lũy)',
    'raw_customer_order_count': 'customer_order_count\n(lần mua, cumcount 0-indexed)',
}
GCOLORS_O = {
    'raw_desc_length': '#2ca02c', 'raw_photos_qty': '#2ca02c',
    'raw_seller_rep': '#9467bd', 'raw_customer_order_count': '#9467bd'
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, var in zip(axes, other_vars):
    means = df.groupby('review_score')[var].mean()
    stds  = df.groupby('review_score')[var].sem()
    c = GCOLORS_O[var]

    ax.plot(means.index, means.values, 'o-', color=c, lw=2.5, ms=9)
    ax.fill_between(means.index,
                    means.values - stds.values,
                    means.values + stds.values,
                    alpha=0.18, color=c)
    for s, v in zip(means.index, means.values):
        ax.text(s, v, f'  {v:.2f}', va='center', fontsize=8.5,
                fontweight='bold', color=c)

    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xlabel('Điểm đánh giá', fontsize=10)
    ax.set_title(NICE_O[var], fontsize=10, fontweight='bold')

plt.suptitle('Hình 3.7 – Nhóm Sản phẩm & Lịch sử: Giá trị TB theo điểm đánh giá',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hinh3_7_sanpham_lichsu_line.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_7_sanpham_lichsu_line.png')

In [ ]:
# ── Hình 3.8: Violin plot – 3 biến giao hàng theo nhãn nhị phân ──────────────
df['Nhãn'] = df['review_binary'].map({0: '0 – Không hài lòng\n(score ≤ 3)',
                                       1: '1 – Hài lòng\n(score > 3)'})
palette_v = {'0 – Không hài lòng\n(score ≤ 3)': '#d62728',
             '1 – Hài lòng\n(score > 3)':        '#2ca02c'}
order_v   = ['0 – Không hài lòng\n(score ≤ 3)', '1 – Hài lòng\n(score > 3)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, var in zip(axes, ['raw_delivery_delay','raw_lead_time','raw_seller_speed']):
    # Clip để violin không quá rộng
    data_v = df[[var, 'Nhãn']].copy()
    lo, hi = data_v[var].quantile([0.02, 0.98])
    data_v = data_v[(data_v[var] >= lo) & (data_v[var] <= hi)]

    sns.violinplot(
        data=data_v, x='Nhãn', y=var,
        order=order_v, palette=palette_v,
        inner='box', ax=ax, linewidth=0.9
    )
    ax.set_xlabel('Nhãn phân loại', fontsize=10)
    ax.set_ylabel(NICE_D[var], fontsize=10)
    ax.set_title(LABEL[var], fontsize=10, fontweight='bold')

plt.suptitle('Hình 3.8 – Violin plot: Biến giao hàng theo review_binary',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hinh3_8_violin_binary.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_8_violin_binary.png')

In [ ]:
# ── Hình 3.9: Phân tích sâu seller_rep theo nhóm + customer_order_count ──────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── seller_rep: bin theo khoảng điểm
df['seller_rep_bin'] = pd.cut(
    df['raw_seller_rep'],
    bins=[1, 2, 3, 3.5, 4, 4.5, 5.01],
    labels=['1–2', '2–3', '3–3.5', '3.5–4', '4–4.5', '4.5–5'],
    right=False
)
srep = df.groupby('seller_rep_bin')['review_score'].agg(['mean','count']).reset_index()
axes[0].bar(
    range(len(srep)), srep['mean'],
    color='#9467bd', alpha=0.82, edgecolor='white', width=0.6
)
for i, row in srep.iterrows():
    axes[0].text(i, row['mean'] + 0.03,
                 f'{row["mean"]:.3f}\n(n={row["count"]:,})',
                 ha='center', fontsize=8.5, fontweight='bold')
axes[0].set_xticks(range(len(srep)))
axes[0].set_xticklabels(srep['seller_rep_bin'], fontsize=9)
axes[0].set_ylabel('Điểm đánh giá trung bình', fontsize=11)
axes[0].set_xlabel('Nhóm seller_rep (danh tiếng người bán)', fontsize=10)
axes[0].set_title('seller_rep theo nhóm → review_score TB', fontsize=10, fontweight='bold')
axes[0].set_ylim(1, 5.5)

# ── customer_order_count: 0 vs 1+
df['loai_khach'] = df['raw_customer_order_count'].apply(
    lambda x: 'Lần đầu mua\n(count = 0)' if x == 0 else 'Mua lại\n(count ≥ 1)'
)
rep_stat = df.groupby('loai_khach')['review_score'].agg(['mean','count'])
rep_stat = rep_stat.reindex(['Lần đầu mua\n(count = 0)', 'Mua lại\n(count ≥ 1)'])

axes[1].bar(
    rep_stat.index, rep_stat['mean'],
    color=['#c5b0d5', '#9467bd'], edgecolor='white', width=0.45, alpha=0.85
)
for i, (idx, row) in enumerate(rep_stat.iterrows()):
    axes[1].text(i, row['mean'] + 0.03,
                 f'{row["mean"]:.3f}\n(n={int(row["count"]):,})',
                 ha='center', fontsize=9, fontweight='bold')
axes[1].set_ylabel('Điểm đánh giá trung bình', fontsize=11)
axes[1].set_xlabel('Loại khách hàng (customer_order_count)', fontsize=10)
axes[1].set_title('customer_order_count → review_score TB', fontsize=10, fontweight='bold')
axes[1].set_ylim(1, 5.5)

plt.suptitle('Hình 3.9 – Phân tích biến Lịch sử: seller_rep và customer_order_count',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hinh3_9_lichsu_phanloai.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_9_lichsu_phanloai.png')

In [ ]:
# ── Hình 3.10: Scatter 4 biến quan trọng nhất (dùng biến norm) ───────────────
top4 = ['norm_delivery_delay', 'norm_lead_time', 'norm_seller_rep', 'norm_freight_ratio']
NICE4 = {
    'norm_delivery_delay':  'norm_delivery_delay',
    'norm_lead_time':       'norm_lead_time',
    'norm_seller_rep':      'norm_seller_rep',
    'norm_freight_ratio':   'norm_freight_ratio',
}

sample = df[top4 + ['review_score']].dropna().sample(
    min(6000, len(df)), random_state=42)
cmap = plt.get_cmap('RdYlGn')

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, feat in zip(axes.flatten(), top4):
    raw_feat = feat.replace('norm_', 'raw_')
    r_val = corr_df[corr_df['Biến'] == raw_feat.replace('raw_','')]['Pearson r'].values
    r_val = r_val[0] if len(r_val) > 0 else float('nan')

    sc = ax.scatter(
        sample[feat], sample['review_score'],
        c=sample['review_score'], cmap=cmap,
        alpha=0.28, s=12, vmin=1, vmax=5
    )
    m, b = np.polyfit(sample[feat], sample['review_score'], 1)
    xs = np.linspace(sample[feat].min(), sample[feat].max(), 200)
    ax.plot(xs, m*xs + b, color='#d62728', lw=2.2, label=f'Pearson r = {r_val:+.3f}')
    ax.set_xlabel(NICE4[feat] + ' (chuẩn hóa 0–1)', fontsize=10)
    ax.set_ylabel('review_score', fontsize=10)
    ax.set_title(NICE4[feat], fontsize=10, fontweight='bold')
    ax.legend(fontsize=9.5, loc='upper right')

plt.suptitle('Hình 3.10 – Scatter plot: 4 biến quan trọng nhất vs. review_score',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hinh3_10_scatter_4bien.png', bbox_inches='tight', dpi=150)
plt.show()
print('✓ Đã lưu: hinh3_10_scatter_4bien.png')

In [ ]:
# ── Kiểm định Kruskal-Wallis: so sánh phân phối biến giữa 5 nhóm điểm ────────
print('KIỂM ĐỊNH KRUSKAL-WALLIS')
print('H0: Phân phối biến không khác nhau giữa 5 nhóm review_score')
print('=' * 65)
for raw in RAW_COLS:
    groups = [df[df['review_score'] == s][raw].dropna() for s in [1, 2, 3, 4, 5]]
    H, p   = kruskal(*groups)
    sig    = '*** BÁC BỎ H0' if p < 0.001 else ('** BÁC BỎ' if p < 0.01 else
              ('* BÁC BỎ' if p < 0.05 else 'Không đủ bằng chứng'))
    print(f'{raw:<30}  H = {H:8.1f}  p = {p:.2e}  {sig}')

In [ ]:
# ── Tổng kết EDA ─────────────────────────────────────────────────────────────
print('=' * 70)
print('TÓM TẮT KẾT QUẢ EDA – CHƯƠNG 3.1')
print('=' * 70)
print(f'\nTổng đơn hàng phân tích: {len(df):,}')
print(f'Phân phối review_score: Mean={df["review_score"].mean():.2f}, Std={df["review_score"].std():.2f}')
print(f'Skewness: {df["review_score"].skew():.4f} → Lệch trái mạnh')
print(f'Tỷ lệ Hài lòng (>3): {(df["review_binary"]==1).mean()*100:.1f}%')
print(f'Tỷ lệ Không hài lòng (≤3): {(df["review_binary"]==0).mean()*100:.1f}%')
print(f'Tỷ lệ mất cân bằng: {(df["review_binary"]==1).sum()}/{(df["review_binary"]==0).sum()} ≈ {(df["review_binary"]==1).mean()/(df["review_binary"]==0).mean():.1f}:1')
print()
print('Top 3 biến tương quan dương mạnh nhất với review_score:')
top_pos = corr_df[corr_df['Pearson r'] > 0].tail(3)
print(top_pos[['Biến','Nhóm','Pearson r','Ý nghĩa']].to_string(index=False))
print()
print('Top 3 biến tương quan âm mạnh nhất với review_score:')
top_neg = corr_df[corr_df['Pearson r'] < 0].head(3)
print(top_neg[['Biến','Nhóm','Pearson r','Ý nghĩa']].to_string(index=False))
print()
ns_vars = corr_df[corr_df['Ý nghĩa'] == 'ns']['Biến'].tolist()
print(f'Biến không có ý nghĩa thống kê: {ns_vars if ns_vars else "Tất cả đều có ý nghĩa"}')
print()
print('Danh sách 10 biểu đồ đã lưu:')
for i, name in enumerate([
    'hinh3_1_phan_phoi_review_score.png',
    'hinh3_2_binary_va_xuhuong.png',
    'hinh3_3_lollipop_tuong_quan.png',
    'hinh3_4_matrix_tuong_quan.png',
    'hinh3_5_giaohang_boxplot.png',
    'hinh3_6_giaodich_bar.png',
    'hinh3_7_sanpham_lichsu_line.png',
    'hinh3_8_violin_binary.png',
    'hinh3_9_lichsu_phanloai.png',
    'hinh3_10_scatter_4bien.png',
], 1):
    print(f'  Hình 3.{i}: {name}')